# File sampling and Using ORCA parser to find energy and 

Run:
```bash
pip install orca-parser
```


Example usage

In [2]:
import orca_parser
from pathlib import Path    

example_path = Path("data_point_0_5.out")

Optimization = orca_parser.ORCAParse(example_path )

Optimization.parse_coords()
print("Atoms:", Optimization.atoms)
print("Final coordinates:")
print(Optimization.coords[0])

Optimization.parse_energies()
print("Final energy:", Optimization.energies[0])

Atoms: ['O', 'O', 'N', 'N', 'N', 'N', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'N', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H', 'H']
Final coordinates:
[[ -2.039189  -1.867224  -0.288869]
 [ -3.366642  -0.557342   0.968367]
 [ -2.743561   1.691281  -0.481914]
 [  2.153539  -0.171898   0.022275]
 [  4.039292  -1.400673  -0.460924]
 [  4.307178   0.818246   0.18353 ]
 [ -0.577635   0.57294   -0.870508]
 [  0.014144   0.928714   0.495562]
 [ -2.114979   0.438057  -0.880322]
 [  1.537726   1.071159   0.437682]
 [ -2.580027  -0.671787   0.049001]
 [  3.435804  -0.223527  -0.071904]
 [ -0.131469  -0.361168  -1.220418]
 [ -0.312262   1.345737  -1.606468]
 [ -0.431033   1.856487   0.869432]
 [ -0.23142    0.140997   1.21941 ]
 [ -2.402752   0.096533  -1.891224]
 [  1.779212   1.898448  -0.259314]
 [  1.891713   1.390808   1.435257]
 [ -3.757064   1.624896  -0.488039]
 [ -2.469399   2.431833  -1.121285]
 [ -2.396191  -2.

In [3]:
def np_to_scfinput(atoms, coords):
    '''
    Input: tuple of (coordinates: np.array; atom representation: tuple of strings)
    '''
    # safeguarding here
    # convertion
    raw = []
    for atom, coord in zip(atoms, coords):
        raw.append(' '.join([atom, *coord.astype(str)]))
    
    return ' ;'.join(raw)

import random
import numpy as np


def random_sampling(SRC, DEST, n_samples=None, seed=42):

    # randomly select n_samples files from SRC directory
    all_files = list(Path(SRC).glob("*.out"))
    if n_samples is not None:
        random.seed(seed)
        sampled_files = random.sample(all_files, n_samples)
    else:
        sampled_files = all_files

    # make DEST directory if it does not exist
    DEST = Path(DEST)
    DEST.mkdir(parents=True, exist_ok=True)

    for file in sampled_files:

        Optimization = orca_parser.ORCAParse(file)

        # parse for coordinate
        Optimization.parse_coords()
        xyz = np_to_scfinput(Optimization.atoms, Optimization.coords[-1])

        # parse for energy
        Optimization.parse_energies()
        energy = Optimization.energies[0]

        # save the data
        np.savez_compressed(
            DEST / f'{file.stem}_xyz_energy.npz',
            energy=energy,
            coordinate=xyz,
        )

    print('done')

testing

In [4]:
random_sampling(SRC='/mnt/i/Dimers_out_only_out_TRAIN', DEST='/mnt/i/sampled_train', n_samples=10000, seed=42)

done


In [5]:
random_sampling(SRC='/mnt/i/Dimers_out_only_out_TEST', DEST='/mnt/i/sampled_test', n_samples=None, seed=42)

done


Inspect files

In [6]:
example = np.load(r"/mnt/d/UROP/pyscf-tutorial/data_sampled/data_point_0_9_xyz_energy.npz")

print(example['energy'].astype(float), '\n',
      example['coordinate'].astype(str))

-115.536964510782 
 O -1.828052 -1.85439 -0.153914 ;O -3.332823 -0.582112 0.930905 ;N -2.583025 1.684485 -0.396305 ;N 2.359772 -0.162014 -0.064338 ;N 4.279907 -1.224281 -0.763191 ;N 4.452993 0.96207 0.008811 ;C -0.428863 0.554017 -0.821179 ;C 0.189104 0.802728 0.555433 ;C -1.966089 0.430041 -0.81365 ;C 1.702921 1.014201 0.474759 ;C -2.452386 -0.683495 0.098993 ;C 3.631243 -0.118191 -0.253195 ;H 0.007483 -0.347641 -1.258289 ;H -0.176685 1.38652 -1.49388 ;H -0.28017 1.676747 1.019769 ;H -0.003878 -0.05839 1.207934 ;H -2.272833 0.103619 -1.825444 ;H 1.889869 1.908301 -0.153504 ;H 2.075121 1.255833 1.487011 ;H -3.575627 1.580419 -0.213955 ;H -2.444275 2.402834 -1.099556 ;H -2.210357 -2.521245 0.440712 ;H 3.719538 -2.035188 -0.963954 ;H 5.271115 -1.255969 -0.923274 ;H 5.438913 0.949292 -0.190163 ;H 4.060807 1.813567 0.370072 ;N -6.046706 -0.619797 2.220155 ;C -7.035336 -1.444982 2.705207 ;C -6.813763 -2.246746 3.844093 ;C -8.298952 -1.500805 2.080964 ;C -7.826045 -3.069266 4.337332 ;C -9.30